# [Chapter 10 — Sensitivity Analysis, §10.4] Quantitative Invasion-Burden Partition

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 10 — Sensitivity Analysis
**Considerations developed:** 6, 10
**Estimated runtime:** ≤ 30s

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/)

## What this notebook does
Quantifies the invasion-burden partition (Theorem 4.X) by computing sensitivity indices for both $\mathcal{R}_0$ and $I^*$ and showing they have different rankings. This translates the qualitative theorem into actionable policy guidance.

## What you should already know
Chapter 10.1-7.3 sensitivity notebooks; Ch 6.3 invasion-burden notebook.


## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import numpy as np
import matplotlib.pyplot as plt
from shared import book_style, BOOK_COLORS, integrate_sir_i
from shared.parameters import baseline_chapter_10
from shared.seeds import set_seed_chapter_10
from shared.verification import assert_within_tolerance
set_seed_chapter_10()
book_style()


In [ ]:
params = baseline_chapter_10()
mu = 1/365
c_I, beta, tau_R = params['c_I'], params['beta'], params['tau_R']
R0 = c_I * beta * tau_R

# Add demographic turnover for endemic equilibrium
def R0_eval(p): return p['c_I'] * p['beta'] * p['tau_R']
def Istar_eval(p):
    R0_ = p['c_I'] * p['beta'] * p['tau_R']
    if R0_ <= 1: return 0
    return (mu / (mu + 1/p['tau_R'])) * (1 - 1/R0_)

names = ['c_I', 'beta', 'tau_R']
S_R0_dict = {n: 1.0 for n in names}  # closed form

def fd_index(eval_func, base, name, h=1e-3):
    y0 = eval_func(base)
    p_plus = base.copy(); p_plus[name] *= (1+h)
    p_minus = base.copy(); p_minus[name] *= (1-h)
    return (base[name]/y0) * (eval_func(p_plus) - eval_func(p_minus)) / (2*h*base[name])
S_Istar_dict = {n: fd_index(Istar_eval, params, n) for n in names}

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(names))
w = 0.35
ax.bar(x - w/2, [S_R0_dict[n] for n in names], w, label='Sensitivity of R_0',
       color=BOOK_COLORS['primary'], alpha=0.85)
ax.bar(x + w/2, [S_Istar_dict[n] for n in names], w, label='Sensitivity of I*',
       color=BOOK_COLORS['infectious'], alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylabel('Normalized sensitivity index')
ax.set_title('Invasion vs burden: different parameters, different rankings')
ax.axhline(0, color='k', lw=0.5)
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nR_0 sensitivities: c_I={S_R0_dict['c_I']:.2f}, beta={S_R0_dict['beta']:.2f}, tau_R={S_R0_dict['tau_R']:.2f}")
print(f"I* sensitivities:  c_I={S_Istar_dict['c_I']:.3f}, beta={S_Istar_dict['beta']:.3f}, tau_R={S_Istar_dict['tau_R']:.3f}")
print()
print("The rankings are NOT identical. A parameter that strongly affects R_0 may")
print("only weakly affect I*, and vice versa. Policy targeting invasion control")
print("(R_0 reduction) is different from policy targeting endemic burden reduction.")
